In [1]:
import cv2
import os
from pathlib import Path
from ultralytics import YOLO

In [2]:
# --- Configuration ---
IMG_DIR = Path("../data/files/not_processed")
OUTPUT_DIR = Path("../data/unlabeled_crops/photos")
MODEL_PATH = "yolo11s.pt"
CLASSIFIER_MODEL_PATH = Path("runs/classifier_convnext/best_model.pth")
FRAMES_PER_VIDEO = 30
VEHICLE_CLASSES = [2, 3, 5, 7] # COCO: car, motorcycle, bus, truck

In [3]:
print(os.listdir(IMG_DIR))


['Files']


In [4]:
CONF = 0.30
PADDING = 10

# Supported image extensions
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".webp"]

# --- Setup ---
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(MODEL_PATH)

# Recursive image search
image_files = []
for ext in IMAGE_EXTS:
    image_files.extend(IMG_DIR.rglob(f"*{ext}"))

print(f"Found {len(image_files)} images.")

Found 14 images.


In [5]:
# --- Main Loop ---
for img_idx, img_path in enumerate(image_files):

    print(f"[{img_idx+1}/{len(image_files)}] Processing {img_path.name}")

    frame = cv2.imread(str(img_path))

    if frame is None:
        print(f"Failed to read {img_path}")
        continue

    h, w, _ = frame.shape

    # YOLO inference
    results = model.predict(
        frame,
        classes=VEHICLE_CLASSES,
        conf=CONF,
        verbose=False
    )

    detection_count = 0

    for result in results:

        boxes = result.boxes.xyxy.cpu().numpy()
        clss = result.boxes.cls.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()

        for det_idx, (box, cls_id, conf) in enumerate(zip(boxes, clss, confs)):

            x1, y1, x2, y2 = map(int, box)

            # Padding
            pad = min(
                PADDING,
                (x2 - x1) // 5,
                (y2 - y1) // 5
            )

            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(w, x2 + pad)
            y2 = min(h, y2 + pad)

            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            # Preserve directory structure
            relative_parent = img_path.parent.relative_to(IMG_DIR)
            out_dir = OUTPUT_DIR / relative_parent
            out_dir.mkdir(parents=True, exist_ok=True)

            # Filename
            crop_name = (
                f"{img_path.stem}"
                f"_d{det_idx}"
                f"_c{int(cls_id)}"
                f"_conf{conf:.2f}.jpg"
            )

            out_path = out_dir / crop_name

            cv2.imwrite(str(out_path), crop)

            detection_count += 1

    print(f"  -> saved {detection_count} crops")

print("Extraction complete.")

[1/14] Processing 1778665567608.jpg
  -> saved 1 crops
[2/14] Processing 1778665567577.jpg
  -> saved 3 crops
[3/14] Processing 1778665567557.jpg
  -> saved 2 crops
[4/14] Processing 1778665567542.jpg
  -> saved 1 crops
[5/14] Processing 1778665567701.jpg
  -> saved 2 crops
[6/14] Processing 1778665567697.jpg
  -> saved 0 crops
[7/14] Processing 1778665567674.jpg
  -> saved 1 crops
[8/14] Processing 1778665567650.jpg
  -> saved 7 crops
[9/14] Processing 1778665567712.jpg
  -> saved 2 crops
[10/14] Processing 1778665567550.jpg
  -> saved 1 crops
[11/14] Processing 1778665567688.jpg
  -> saved 1 crops
[12/14] Processing 1778665567566.jpg
  -> saved 6 crops
[13/14] Processing 1778665567659.jpg
  -> saved 1 crops
[14/14] Processing 1778665567589.jpg
  -> saved 2 crops
Extraction complete.
